# MLOps Tutorial: Handling Data Drift in Production

Welcome! This notebook will teach you about **MLOps** (Machine Learning Operations) - the practice of deploying and maintaining machine learning models in production.

## What You'll Learn

1. **Data Drift**: How data changes over time in production
2. **Model Retraining**: When and how to update models
3. **Deployment Strategies**: Safe ways to roll out new models
4. **Champion/Challenger Pattern**: Testing new models against production models
5. **Canary Releases**: Gradual rollouts to minimize risk

## The Scenario

Imagine you're managing a machine learning system that predicts customer behavior. Over time:
- Customer preferences change
- Market conditions shift
- Your model's accuracy degrades

**Your challenge**: Keep the system performing well despite these changes!

---

Let's get started! 🚀

## Part 1: Setup and Imports

First, we'll import the libraries we need and set up our environment.

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Import utility functions
from utils import (generate_batch, drift_schedule, plot_drift_over_time,
                   plot_model_accuracies, plot_served_accuracy, plot_sample_data)

# Set random seed for reproducibility
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)

print("✓ Setup complete!")

## Part 2: Configuration Parameters

These parameters control how our simulation runs. You can experiment with these later!

### What do these mean?

**Simulation Settings:**
- `ROUNDS`: How many time periods we'll simulate (like days or weeks)
- `BATCH_SIZE`: How many predictions we make each round

**Model B Retraining:**
- `RETRAINING_FREQUENCY`: How often we retrain Model B (every N rounds)
- `FRESHNESS_RELEVANCE`: How much we prioritize recent data (0.0 = treat all data equally, 1.0 = only focus on new data)

**Promotion Controls:**
- `PROMOTION_THRESHOLD`: How much better a model must be to get promoted (2% accuracy)
- `PROMOTION_PATIENCE`: How many rounds in a row it must be better

**Canary Release:**
- `CANARY_FRACTION`: What % of traffic goes to the challenger (20%)

In [ ]:
# --- Simulation parameters ---
ROUNDS = 50              # number of deployment rounds
BATCH_SIZE = 400         # number of samples per round

# --- Model B (retraining) controls ---
RETRAINING_FREQUENCY = 5   # how often to retrain (in rounds)
FRESHNESS_RELEVANCE = 0.5  # 0.0 = keep old data dominant, 1.0 = prioritize newest data heavily

# --- Promotion controls ---
PROMOTION_THRESHOLD = 0.02   # B must beat A by this absolute accuracy margin
PROMOTION_PATIENCE = 3       # number of consecutive rounds meeting threshold to promote

# --- Canary controls ---
CANARY_FRACTION = 0.20       # share of traffic served by challenger each round

print(f"✓ Configuration set: {ROUNDS} rounds, retraining every {RETRAINING_FREQUENCY} rounds")

## Part 3: Understanding the Data

Our dataset is a **binary classification problem** - we're predicting one of two outcomes (0 or 1).

Think of it like:
- Will a customer buy? (Yes/No)
- Is this transaction fraud? (Yes/No)
- Will a user click the ad? (Yes/No)

The data has **two classes**:
- **Class 0**: Stays in roughly the same place (stationary)
- **Class 1**: **Drifts over time** - this simulates how real-world data changes!

Let's generate and visualize a sample batch:

In [ ]:
# Generate a sample batch with no drift
X_sample, y_sample = generate_batch(mu_shift=0.0, size=BATCH_SIZE)

print(f"Sample shape: {X_sample.shape}")
print(f"Features (X): {X_sample.shape[1]} dimensions")
print(f"Labels (y): {len(y_sample)} samples")
print(f"Class distribution: {np.sum(y_sample == 0)} class 0, {np.sum(y_sample == 1)} class 1")

# Visualize the data using our utility function
plot_sample_data(X_sample, y_sample)

## Part 4: Initial Training - YOUR TASK! 🎯

Now it's your turn! You need to train both models.

### Your Tasks:
1. Train Model B (the challenger)
2. Calculate baseline accuracy for Model B

In [ ]:
# Step 1: Generate initial training data (no drift)
X_init, y_init = generate_batch(mu_shift=0.0, size=2000)

print(f"✓ Generated {len(y_init)} training samples")

In [ ]:
# Step 2: Train Model A (Champion) - PROVIDED FOR YOU
model_A = LogisticRegression(max_iter=1000)
model_A.fit(X_init, y_init)

print("✓ Model A trained!")

In [ ]:
# Step 3: Train Model B (Challenger) - YOUR CODE HERE! 🎯
# Model B should start with the same training as Model A

# HINT: Create a LogisticRegression model (same as Model A)
# HINT: Use max_iter=1000 as the parameter
# HINT: Fit it on X_init and y_init (same data as Model A)

model_B = # YOUR CODE HERE
# YOUR CODE HERE to fit the model

print("✓ Model B trained (same initial data as Model A)")

In [ ]:
# Initialize Model B's training history
# We'll keep track of all the data B has seen for future retraining

X_hist = X_init.copy()
y_hist = y_init.copy()
w_hist = np.ones_like(y_hist, dtype=float)  # sample weights (all equal initially)

print(f"✓ Training history initialized with {len(y_hist)} samples")

In [ ]:
# Step 4: Evaluate both models on a holdout set - YOUR CODE HERE! 🎯
X_hold, y_hold = generate_batch(mu_shift=0.0, size=1000)

# Compute baseline accuracy for Model A - PROVIDED FOR YOU
base_acc_A = accuracy_score(y_hold, model_A.predict(X_hold))

# Compute baseline accuracy for Model B - YOUR CODE HERE!
# HINT: Follow the same pattern as Model A above
# HINT: Use accuracy_score(y_true, y_pred)
# HINT: Get predictions using model_B.predict(X_hold)

base_acc_B = # YOUR CODE HERE

print(f"\n{'='*60}")
print(f"BASELINE PERFORMANCE (No Drift)")
print(f"{'='*60}")
print(f"Model A (Champion): {base_acc_A:.3f}")
print(f"Model B (Challenger): {base_acc_B:.3f}")
print(f"{'='*60}\n")

### ✅ Check Your Work

Before proceeding, verify:
- Both models should have the **same** baseline accuracy (they're identical right now)
- Accuracy should be somewhere between 0.80 and 0.95 (this is a relatively easy problem)

If something looks wrong, go back and check your code!

## Part 5: Going Live - Deployment Simulation

Now we'll simulate what happens in production over 50 rounds. Each round represents a time period (day, week, etc.).

### What happens each round?

1. **Data drifts** - Class 1 shifts to a new location
2. **New data arrives** - We get a fresh batch of predictions to make
3. **Canary routing** - Split traffic between champion and challenger
4. **Evaluate performance** - How well do the models do?
5. **Check for promotion** - Should we switch champions?
6. **Retrain Model B** - If it's time, update Model B with new data

Let's set up the tracking variables first:

In [ ]:
# Initialize tracking lists
drift_values = []           # Track how much drift has occurred
acc_A_hist = []             # Model A's accuracy each round
acc_B_hist = []             # Model B's accuracy each round
served_acc_hist = []        # Blended accuracy (what users experience)
retrain_rounds = []         # Which rounds we retrained Model B
promotions = []             # Which rounds we promoted Model B

# Model A is always the champion (static model)
# Model B is always the challenger (retrained model being tested)
promoted = False            # Track if B has been promoted to replace A
consecutive_wins = 0        # How many consecutive rounds B has beaten A

mu = 0.0  # Starting drift (no drift yet)

print("✓ Tracking variables initialized")
print(f"Champion: Model A (static)")
print(f"Challenger: Model B (retrained)")

## Part 6: The Main Simulation Loop - YOUR TASK! 🎯

This is the heart of the simulation. You'll implement key parts of the evaluation and promotion logic.

### The Setup:
- **Model A (Champion)**: Static model, never retrained, serves most traffic (80%)
- **Model B (Challenger)**: Retrained model, serves canary traffic (20%), being tested for promotion

### Your Tasks in the Loop:
1. Get predictions from Model B on canary traffic
2. Calculate canary accuracy
3. Evaluate Model B's full performance
4. Track Model B's accuracy history
5. Implement the promotion logic

In [ ]:
# Main simulation loop
for r in range(1, ROUNDS + 1):
    # 1) Update drift
    mu += drift_schedule(r, ROUNDS)
    drift_values.append(mu)
    
    # 2) Generate live batch with current drift
    X_live, y_live = generate_batch(mu_shift=mu, size=BATCH_SIZE)
    
    # 3) Split batch for realistic canary routing
    canary_size = int(BATCH_SIZE * CANARY_FRACTION)
    X_canary = X_live[:canary_size]      # Canary traffic (20%) → Model B (challenger)
    y_canary = y_live[:canary_size]
    X_champion = X_live[canary_size:]    # Champion traffic (80%) → Model A (champion)
    y_champion = y_live[canary_size:]
    
    # 4) Each model serves its assigned traffic
    # Model A (champion) serves most traffic (80%) - PROVIDED FOR YOU
    y_pred_champion = model_A.predict(X_champion)
    acc_champion = accuracy_score(y_champion, y_pred_champion)
    
    # Model B (challenger) serves canary traffic (20%) - YOUR CODE HERE! 🎯
    # HINT: Use model_B.predict(X_canary) to get predictions
    # HINT: Use accuracy_score(y_canary, y_pred_canary) to calculate accuracy
    
    y_pred_canary = # YOUR CODE HERE
    acc_canary = # YOUR CODE HERE
    
    # 5) For tracking full model performance (on entire batch)
    # Model A's full performance - PROVIDED FOR YOU
    aA = accuracy_score(y_live, model_A.predict(X_live))
    
    # Model B's full performance - YOUR CODE HERE! 🎯
    # HINT: Follow the same pattern as Model A above
    aB = # YOUR CODE HERE
    
    # Store full accuracies for plotting
    acc_A_hist.append(aA)
    # YOUR CODE HERE: Append aB to acc_B_hist
    
    # 6) Calculate served accuracy (what users actually experience)
    served_acc = (1 - CANARY_FRACTION) * acc_champion + CANARY_FRACTION * acc_canary
    served_acc_hist.append(served_acc)
    
    # 7) Promotion logic: Check if B (challenger) should replace A (champion)
    # YOUR CODE HERE! 🎯
    if not promoted:  # Only check for promotion if B hasn't been promoted yet
        # TASK: Implement the promotion logic
        # HINT: Check if (aB - aA) >= PROMOTION_THRESHOLD
        # HINT: If yes, increment consecutive_wins
        # HINT: If no, reset consecutive_wins to 0
        # HINT: If consecutive_wins >= PROMOTION_PATIENCE, set promoted = True
        # HINT: When promoting, also append r to promotions and print a message
        
        # YOUR CODE HERE
        
    # 8) Retrain Model B on schedule - PROVIDED FOR YOU
    if r % RETRAINING_FREQUENCY == 0:
        # Decay old sample weights
        w_hist *= (1.0 - FRESHNESS_RELEVANCE)
        
        # Add new data with fresh weights
        X_hist = np.vstack([X_hist, X_live])
        y_hist = np.concatenate([y_hist, y_live])
        w_new = np.full(shape=y_live.shape, fill_value=FRESHNESS_RELEVANCE, dtype=float)
        w_hist = np.concatenate([w_hist, w_new])
        
        # Retrain Model B with weighted data
        model_B.fit(X_hist, y_hist, sample_weight=w_hist)
        retrain_rounds.append(r)
        print(f"🔄 Round {r}: Model B retrained (total data: {len(y_hist)} samples)")

print(f"\n{'='*60}")
print(f"SIMULATION COMPLETE")
print(f"{'='*60}")
print(f"Model B promoted: {'Yes' if promoted else 'No'}")
if promoted and len(promotions) > 0:
    print(f"Promoted at round: {promotions[0]}")
print(f"Total retraining events: {len(retrain_rounds)}")
print(f"{'='*60}\n")

## Part 7: Visualization and Analysis

Now let's visualize what happened during the simulation!

### Plot 1: Data Drift Over Time

This shows how much the data distribution has shifted from the original.

In [ ]:
plot_drift_over_time(drift_values, ROUNDS)

### Plot 2: Model Accuracies Over Time

This shows how each model performed:
- **Dotted vertical lines (`:`)**: When Model B was retrained
- **Dashed vertical lines (`--`)**: When champion was promoted

**Watch for:**
- Does Model A's accuracy decline as drift increases?
- Does Model B maintain better accuracy after retraining?
- Do promotions happen when Model B is clearly better?

In [ ]:
plot_model_accuracies(acc_A_hist, acc_B_hist, retrain_rounds, promotions, ROUNDS)

### Plot 3: Served Accuracy (What Users Experience)

This shows the **blended accuracy** from canary routing - what your users actually experience.

Notice it's often **smoother** than individual model performance because we're blending predictions.

In [ ]:
plot_served_accuracy(served_acc_hist, ROUNDS)

## Part 8: Summary Statistics

Let's calculate some key metrics to understand performance:

In [ ]:
# Calculate average accuracies
avg_acc_A = np.mean(acc_A_hist)
avg_acc_B = np.mean(acc_B_hist)
avg_served = np.mean(served_acc_hist)

# Calculate final accuracies (last 10 rounds)
final_acc_A = np.mean(acc_A_hist[-10:])
final_acc_B = np.mean(acc_B_hist[-10:])
final_served = np.mean(served_acc_hist[-10:])

# Calculate accuracy degradation for Model A
initial_acc_A = np.mean(acc_A_hist[:5])
degradation = initial_acc_A - final_acc_A

print(f"\n{'='*60}")
print(f"PERFORMANCE SUMMARY")
print(f"{'='*60}")
print(f"\nAverage Accuracy (all rounds):")
print(f"  Model A (static):     {avg_acc_A:.3f}")
print(f"  Model B (retrained):  {avg_acc_B:.3f}")
print(f"  Served (blended):     {avg_served:.3f}")
print(f"\nFinal Accuracy (last 10 rounds):")
print(f"  Model A (static):     {final_acc_A:.3f}")
print(f"  Model B (retrained):  {final_acc_B:.3f}")
print(f"  Served (blended):     {final_served:.3f}")
print(f"\nModel A Degradation:")
print(f"  Initial: {initial_acc_A:.3f} → Final: {final_acc_A:.3f}")
print(f"  Loss: {degradation:.3f} ({degradation*100:.1f}% points)")
print(f"\nMLOps Events:")
print(f"  Retraining events:  {len(retrain_rounds)}")
print(f"  Model B promoted:   {'Yes' if promoted else 'No'}")
if promoted and len(promotions) > 0:
    print(f"  Promoted at round:  {promotions[0]}")
print(f"{'='*60}\n")

## Part 9: Reflection Questions 🤔

Take a moment to think about what you've learned:

### Questions to Consider:

1. **Why did Model A's accuracy decline?**

2. **How did retraining help Model B?**

3. **Why use canary releases instead of immediately switching?**

4. **What if we retrained more frequently?**

5. **What does FRESHNESS_RELEVANCE control?**

---

**Write your answers here:**

*(Double-click to edit this cell)*

**Answer 1:**

**Answer 2:**

**Answer 3:**

**Answer 4:**

**Answer 5:**

## Congratulations! 🎉

You've completed the MLOps tutorial and learned about:
- Data drift and its impact on model performance
- Champion/Challenger deployment patterns
- Canary releases for safe rollouts
- Model retraining strategies
- Promotion criteria and safety mechanisms

These are fundamental concepts used by companies like Netflix, Uber, Amazon, and Google to keep their ML systems healthy in production!